# Stage1 TSMC 季度指引原始来源预检
仅验证三份已冻结页面响应，不读市场结果。

In [1]:
import hashlib,json,shlex,subprocess,tarfile,tempfile
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd().resolve();card=root/'research/stage1_tsmc_quarterly_guidance_source_preflight_20261023'
plan=json.loads((card/'plan.json').read_text());receipt=json.loads((card/'capture_receipts.json').read_text())
assert receipt['plan_sha256']==hashlib.sha256((card/'plan.json').read_bytes()).hexdigest()
assert len(receipt['captures'])==plan['request_budget']['GETs']==3
entries=[]
for row in receipt['captures']:
    assert row['url']==plan['exact_urls'][row['index']-1]
    assert row['curl_exit']==0 and row['http_status']=='403'
    for kind in ('body','headers'):
        entry=row[kind];path=root/entry['path'];blob=path.read_bytes()
        assert len(blob)==entry['bytes'] and hashlib.sha256(blob).hexdigest()==entry['sha256']
        entries.append({'path':entry['path'],'bytes':len(blob),'sha256':entry['sha256']})
body_hashes=[row['body']['sha256'] for row in receipt['captures']]
assert len(body_hashes)==3
paths=[card/'plan.json',card/'capture_receipts.json']+[root/x['path'] for x in entries]
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='tsmc-guidance-source-') as directory:
    archive=Path(directory)/'source.tar.gz'
    with tarfile.open(archive,'w:gz') as output:
        for path in paths:output.add(path,arcname=str(path.relative_to(root)),recursive=False)
    digest=hashlib.sha256(archive.read_bytes()).hexdigest()
    evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
    assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
    remote_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
    remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(remote_path)).split()[0]
    assert remote==digest
    restored=Path(directory)/'restored.tar.gz'
    with restored.open('wb') as out:
        transfer=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote_path)],stdout=out,stderr=subprocess.PIPE,timeout=60)
    assert transfer.returncode==0 and hashlib.sha256(restored.read_bytes()).hexdigest()==digest
    with tarfile.open(restored,'r:gz') as source:
        members={m.name:m for m in source.getmembers()}
        assert set(members)=={str(p.relative_to(root)) for p in paths}
        for path in paths:
            name=str(path.relative_to(root));data=source.extractfile(members[name]).read()
            assert data==path.read_bytes()
    evidence['nas_path']=remote_path
summary={'experiment':plan['experiment'],'decision':'needs_source_access_three_frozen_official_pages_HTTP403','objective_achieved':False,'plan_sha256':receipt['plan_sha256'],'official_page_GETs':3,'all_status_403':True,'body_sha256_by_anchor':body_hashes,'retrieved_original_issuer_guidance_text':False,'historical_first_public_qualified':False,'model_fits':0,'native_accounts':0,'market_outcome_reads':0,'next_action':'Close this exact three-URL direct transport version; a materially different authorized source route must supply original issuer bytes and publication timing before any Stage1 feature or information screen.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]},'nas_evidence':evidence,'nas_scoped_restore_verified':True}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
(card/'conclusion.md').write_text('# TSMC 季度前瞻指引来源预检\n\n冻结三份发行人英文新闻页（2011-04-28、2018-10-18、2022-07-14），各只请求一次；三次均返回 HTTP 403，三份响应正文 SHA-256 各自记录。原文指引数值、币种、未来季度与首次发布时间均未从本机原始响应获得，因此本固定直连来源版本为 `needs_source_access`。\n\n三份响应和响应头的原始字节、请求时刻、URL 与哈希已保存，并上传 Yanxuan NAS；远端哈希和限定恢复逐文件通过。0 市场收益读取、0 拟合、0 账户。网页搜索中的当前页面内容不能替代当年的首发字节；不生成因子或 SOXX 成功结论。下一步只可通过另一个事先冻结、能提供原始来源及发布时间的渠道继续。\n')
print(summary['decision'],summary['nas_evidence']['sha256'])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


needs_source_access_three_frozen_official_pages_HTTP403 24a098f7411cb28f3ca2aa282a8dd416ec1c649bf3a5bc9a0d4a4a17c21276fd


## 结论
三份发行人页面在本机均返回 403；原始指导数值与首次公开时点未合格。无模型或账户。